# 📘 Notebook 17: Pretrained Models & Transfer Learning

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module E: Transformers & Fine-Tuning · Notebook 2 of 3 in this module · (17 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain **pretraining** and why it changed modern AI
- Understand **transfer learning** and when to use it
- Distinguish **feature extraction** from **fine-tuning**
- Use the **Hugging Face** ecosystem to load and run a pretrained model
- Apply a pretrained model to a real task with only a few lines of code

> **Prerequisites:** Notebook 16 (Transformers).
>
> 🔭 **A shift in mindset.** Until now we built and trained models ourselves. In practice, almost no one trains a large model from scratch, it costs enormous data and compute. Instead we **reuse** models that others have already trained, adapting them to our task. This is the most practically important skill in the whole course.

> ℹ️ **Setup note.** This notebook uses Hugging Face `transformers`. In a browser kernel, downloading model weights may be impractical. Read the code carefully, it is exactly what you would run in a standard environment (Colab, a server, or a local machine):
```python
import piplite
await piplite.install(['transformers', 'torch'])
```

## 1. Pretraining: learning general knowledge first

### The idea
**Pretraining** trains a large model on a massive, general dataset before it ever sees your specific task. A language model is pretrained on enormous amounts of text, learning grammar, facts, and reasoning patterns; a vision model is pretrained on millions of images, learning edges, textures, shapes, and objects. The result is a model with broad, reusable knowledge.

### Why it changed everything
Pretraining is expensive, often millions of dollars of compute, but it happens **once**. Everyone else can then download the pretrained model and adapt it cheaply. This division of labour is why a student today can build on a model that would have been impossible to train alone.

## 2. Transfer learning

### Intuition first
Someone who already plays the piano learns the organ quickly, they transfer most of their skill and only adapt the differences. **Transfer learning** applies the same principle: take a model that has learned general features on a large task, and reuse that knowledge for a new, related task with far less data and time.

### Why it works
The early layers of a deep network learn **general** features (recall the feature hierarchy from Notebook 14: edges, then shapes, then objects). These general features are useful across many tasks. Only the later, task-specific layers need to change. So we keep the general knowledge and re-train only what is necessary.

### Two strategies

| Strategy | What you do | When to use |
|----------|-------------|-------------|
| **Feature extraction** | Freeze the pretrained model; train only a small new layer on top | Little data; fast; safe |
| **Fine-tuning** | Continue training some or all of the pretrained weights on your data | More data; need higher accuracy |

Feature extraction treats the pretrained model as a fixed 'feature generator'. Fine-tuning (the focus of Notebook 18) gently adjusts the model's own weights to your task. The right choice depends mainly on how much labelled data you have.

> 🧠 **A rule of thumb.** Less data → lean toward feature extraction (fewer things to learn, less risk of overfitting). More data → fine-tuning can pay off. With a medical-imaging dataset of a few hundred scans, for instance, feature extraction from a model pretrained on millions of images is often the wiser starting point.

## 3. The Hugging Face ecosystem

**Hugging Face** is the de facto hub for pretrained models. Its `transformers` library lets you download and run thousands of models with a few lines of code. The quickest entry point is the **pipeline**, which bundles all the steps (tokenising the input, running the model, decoding the output) behind a single call.

In [ ]:
# Sentiment analysis with a pretrained model -- no training needed
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

results = classifier([
    "This course made deep learning finally click for me.",
    "I am completely lost and frustrated.",
])
for r in results:
    print(r)

In three lines we used a Transformer (the architecture from Notebook 16), pretrained on a vast corpus, to classify sentiment, a task we never trained for. That is the power of reuse.

### Tokenisation: how text becomes numbers
Models do not read text; they read numbers. A **tokeniser** splits text into tokens (words or word-pieces) and maps each to an integer ID. Every pretrained model comes with its own matching tokeniser, which you must use:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Transformers are powerful."
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token IDs:", ids)

These integer IDs are what flow into the model's first layer (an *embedding* that turns each ID into a vector, on which attention then operates). Notice 'Transformers' may split into word-pieces, this lets the model handle any word, even ones it never saw, by composing it from familiar pieces.

> ⚠️ **Common pitfalls**
>
> - Always pair a model with **its own** tokeniser (`from_pretrained` with the same name). A mismatched tokeniser produces meaningless inputs.
> - Pretrained models can carry **biases** from their training data. For any real application, especially sensitive ones like medicine, evaluate carefully and never assume a pretrained model is neutral or correct.

## 4. Transfer learning for images (conceptual walkthrough)

The same idea applies in vision. To classify a small custom image dataset, you would:
1. load a model pretrained on a large image corpus (e.g. a ResNet, the CNN family from Notebook 14);
2. **freeze** its convolutional layers, keep their learned general features;
3. replace its final classification layer with a new one sized for *your* classes;
4. train only that new layer on your data.

The code skeleton (for a standard environment) looks like this:

In [ ]:
# Conceptual example -- typical pattern with torchvision
import torch
import torch.nn as nn
from torchvision import models

# 1. Load a model pretrained on ImageNet
resnet = models.resnet18(weights="IMAGENET1K_V1")

# 2. Freeze all existing parameters (feature extraction)
for param in resnet.parameters():
    param.requires_grad = False

# 3. Replace the final layer for our own number of classes (say 3)
n_features = resnet.fc.in_features
resnet.fc = nn.Linear(n_features, 3)   # only THIS layer will train

# 4. Now train only resnet.fc on your data, with the usual loop from Notebook 13
print("Ready for transfer learning. Trainable layer:", resnet.fc)

> 🔭 **Everything connects.** Freezing layers, replacing the final layer, and training with the standard loop all rest on concepts you built earlier: layers (Notebook 12), the training loop (Notebook 13), CNNs (Notebook 14). Transfer learning is not a new black box, it is a smart *reuse* of the machinery you already understand.

---
## ✏️ Exercises

*If model downloads are unavailable in your environment, answer the conceptual exercises in a text cell.*

### Exercise 1
You have **80** labelled chest X-ray images and want to classify them as normal vs abnormal. Would you choose feature extraction or full fine-tuning, and why? (Two or three sentences.)

In [ ]:
# Your answer here:


<details><summary>💡 Show solution</summary>

```python
# With only 80 images, FEATURE EXTRACTION is the safer choice. Fine-tuning millions
# of weights on so little data risks severe overfitting. Freezing a model pretrained
# on large image data and training only a small new classification head reuses general
# visual features while keeping the number of newly-learned parameters tiny.
```

*This mirrors the 'less data -> feature extraction' rule of thumb and the medical-imaging note above.*
</details>

### Exercise 2
Use the Hugging Face `pipeline` for a different task, for example `pipeline("zero-shot-classification")` or `pipeline("summarization")`, on a sentence of your choice. (Run in a suitable environment.) Describe what task it performs.

In [ ]:
from transformers import pipeline
# Your experiment here:


<details><summary>💡 Show solution</summary>

```python
summariser = pipeline("summarization")
text = "..."  # a few sentences of your choosing
print(summariser(text, max_length=40, min_length=10))
```

*The pipeline abstracts tokenisation, model inference, and decoding. Each task name loads an appropriate pretrained model.*
</details>

## 🔑 Key takeaways

- **Pretraining** teaches a model broad knowledge on a massive dataset, once, at great cost.
- **Transfer learning** reuses that knowledge for a new task with far less data and compute.
- **Feature extraction** (freeze + train a new head) suits small datasets; **fine-tuning** suits larger ones.
- **Hugging Face** `pipeline` runs powerful pretrained Transformers in a few lines; always use a model's matching **tokeniser**.
- Transfer learning reuses the exact machinery (layers, training loop, CNNs) you built earlier, not a new black box.

---
**Next:** *Notebook 18: Fine-Tuning a Model*, the course finale, where you adapt a pretrained model to your own data end to end.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*